In [1]:
import polars as pl
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
DATASET_PATH = "/group/pmc021/amunif/epi-thesis/workflow/13_Healthy liver with manual preprocessing/dataset/donor_3"

In [3]:
def load_histone(filename):
    # Create a schema
    intersect_schema = pl.Schema({
        # The E066.bed files
        'chromosome_name': pl.String,
        'start': pl.Int64,
        'end': pl.Int64,
        'gene_id': pl.String,
        'E066': pl.Float64,
        'strand': pl.String,
        'label': pl.String,
        'external_gene_name': pl.String,
        'start_position': pl.Int64,
        'end_position': pl.Int64,
        'tss': pl.Int64,

        # The gappedPeak column
        "chrom_p": pl.String,
        "chromStart_p": pl.Int64, 
        "chromEnd_p": pl.Int64, 
        "name_p": pl.String, 
        "score_p": pl.Float64, 
        "strand_p": pl.String,
        "thickStart": pl.Int64,
        "thickEnd": pl.Int64,
        "itemRgb": pl.Int64,
        "blockCount": pl.Int64,
        "blockSizes": pl.String,
        "blockStarts": pl.String,
        "signalValue": pl.Float64,
        "pValue": pl.Float64,
        "qValue": pl.Float64
    })

    # Open file
    histone_df = pl.read_csv(
            filename,
            separator="\t",
            has_header = False,
            schema = intersect_schema   
        )
    
    return histone_df

In [4]:
def load_gene_expression(filename):
    # Create a schema
    donor_schema = pl.Schema({
        'chromosome_name': pl.String,
        'start': pl.Int64,
        'end': pl.Int64,
        'gene_id': pl.String,
        'value': pl.Float64,
        'strand': pl.Int64,
        'label': pl.String,
        'external_gene_name': pl.String,
        'start_position': pl.Int64,
        'end_position': pl.Int64,
        'tss': pl.Int64
    })

    # Read the file
    df = pl.read_csv(filename, has_header=False, schema=donor_schema, separator="\t")

    return df
    

In [5]:
def create_empty_dataframe(histone_name):
    schema = pl.Schema({
        'gene_id': pl.String,
        histone_name: pl.List(pl.Float64),
        f'{histone_name}_wc': pl.UInt32,
        f'{histone_name}_len': pl.UInt32
    })

    df = pl.DataFrame(schema=schema)

    return df

In [6]:
def build_matrix(genes_df, histone_df, histone_name):
    # Build the dataframe with window
    genes_with_windows = genes_df.with_columns([
        pl.int_ranges(pl.col('start'), pl.col('end'), 100).alias('window_start')
    ]).explode('window_start')

    genes_with_windows = genes_with_windows.with_columns([
            (pl.col('window_start') + 100).alias('window_end')
    ])

    # Join genes with histone data
    joined_df = genes_with_windows.join(
        histone_df,
        left_on='gene_id',
        right_on='gene_id',
        how='left'
    )

    # Filter and calculate average signal value
    result_df = joined_df.filter(
        (pl.col('chromStart_p') < pl.col('window_end')) &
        (pl.col('chromEnd_p') > pl.col('window_start'))
    ).group_by(['gene_id', 'window_start'], maintain_order=True).agg([
        pl.col('signalValue').mean().alias(histone_name)
    ]).sort(['gene_id', 'window_start'])

    # Find the gene without histone match
    genes_wo_histone = genes_with_windows.join(
        result_df,
        on=["gene_id", "window_start"],
        how="anti"
    )

    # Add the signalValue column so it can be merged
    genes_wo_histone = genes_wo_histone.with_columns(
        signalValue = pl.lit(0.0).cast(pl.Float64)
    )

    # Aggregate the genes without histone result
    genes_wo_histone = genes_wo_histone.group_by(['gene_id', 'window_start'], maintain_order=True).agg([
            pl.col('signalValue').mean().alias(histone_name)
        ]).sort(['gene_id', 'window_start'])

    # Merge both (results and genes without histone)
    result_df.extend(genes_wo_histone)

    # Sort the result dataframe by gene_id and window start for aggregation
    sorted_result_df = result_df.sort(['gene_id', 'window_start'])
    
    # Group by to make array of features
    matrix_df = (
        sorted_result_df
        .with_columns(pl.col(histone_name).fill_null(0))
        .group_by(['gene_id'], maintain_order=True)
        .agg(pl.col(histone_name))
        .sort('gene_id')
    )

    # Add the count and length of array feature for checking
    matrix_control_df = matrix_df.with_columns(
        pl.col(histone_name)
        .list.eval(pl.element().is_not_null() & (pl.element() > 0))
        .list.sum()
        .alias(f"{histone_name}_wc")
    )

    matrix_control_df = matrix_control_df.with_columns(
        pl.col(histone_name).list.len().alias(f"{histone_name}_len")
    )

    # Finally, return the gene_id with its histone features
    return matrix_control_df

In [7]:
# Getting histone in chunk
def get_histone_features(genes_df, histone_df, histone_name):
    
    genes_w_histone = create_empty_dataframe(histone_name)
    
    for i, chunk in enumerate(genes_df.iter_slices(n_rows=100)):
        if i % 100 == 0:
            print(f"Processing {histone_name}: {i*100}/{genes_df.height}")
        
        result = build_matrix(chunk, histone_df, histone_name)
        genes_w_histone.extend(result)

    print(f"Processing {histone_name} features finished.")
    return genes_w_histone

In [8]:
# Load the healthy liver expression file
genes_df = load_gene_expression(os.path.join(DATASET_PATH, "../healthy_liver.bed"))

In [9]:
genes_df

chromosome_name,start,end,gene_id,value,strand,label,external_gene_name,start_position,end_position,tss
str,i64,i64,str,f64,i64,str,str,i64,i64,i64
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,"""1""","""TSPAN6""",99883667,99894988,99894988
"""chrX""",99834799,99844799,"""ENSG00000000005""",0.191,1,"""0""","""TNMD""",99839799,99854882,99839799
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,-1,"""1""","""DPM1""",49551404,49575092,49575092
"""chr1""",169858408,169868408,"""ENSG00000000457""",4.733,-1,"""1""","""SCYL3""",169818772,169863408,169863408
"""chr1""",169626245,169636245,"""ENSG00000000460""",0.942,1,"""0""","""C1orf112""",169631245,169823221,169631245
…,…,…,…,…,…,…,…,…,…,…
"""chr15""",102280913,102290913,"""ENSG00000259658""",0.212,-1,"""0""","""RP11-89K11.1""",102277302,102285913,102285913
"""chr15""",97966182,97976182,"""ENSG00000259664""",0.0,-1,"""0""","""CTD-2147F2.2""",97913601,97971182,97971182
"""chr16""",33642696,33652696,"""ENSG00000259680""",0.071,-1,"""0""","""RP11-812E19.9""",33647044,33647696,33647696


In [10]:
# Load the histone dataset
H3K9ac_df = load_histone(os.path.join(DATASET_PATH, 'H3K9ac.bed'))
H3K9me3_df = load_histone(os.path.join(DATASET_PATH, 'H3K9me3.bed'))
H3K4me3_df = load_histone(os.path.join(DATASET_PATH, 'H3K4me3.bed'))
H3K27ac_df = load_histone(os.path.join(DATASET_PATH, 'H3K27ac.bed'))
H3K27me3_df = load_histone(os.path.join(DATASET_PATH, 'H3K27me3.bed'))

In [11]:
# Build the matrix
genes_w_H3K9ac_df = get_histone_features(genes_df, H3K9ac_df, 'H3K9ac')
genes_w_H3K9me3_df = get_histone_features(genes_df, H3K9me3_df, 'H3K9me3')
genes_w_H3K4me3_df = get_histone_features(genes_df, H3K4me3_df, 'H3K4me3')
genes_w_H3K27ac_df = get_histone_features(genes_df, H3K27ac_df, 'H3K27ac')
genes_w_H3K27me3_df = get_histone_features(genes_df, H3K27me3_df, 'H3K27me3')

Processing H3K9ac: 0/19645
Processing H3K9ac: 10000/19645
Processing H3K9ac features finished.
Processing H3K9me3: 0/19645
Processing H3K9me3: 10000/19645
Processing H3K9me3 features finished.
Processing H3K4me3: 0/19645
Processing H3K4me3: 10000/19645
Processing H3K4me3 features finished.
Processing H3K27ac: 0/19645
Processing H3K27ac: 10000/19645
Processing H3K27ac features finished.
Processing H3K27me3: 0/19645
Processing H3K27me3: 10000/19645
Processing H3K27me3 features finished.


In [12]:
# Checking the generate features
genes_w_H3K9ac_df.filter(pl.col('H3K9ac_wc') > 0).sort(['H3K9ac_wc'], descending=True)

gene_id,H3K9ac,H3K9ac_wc,H3K9ac_len
str,list[f64],u32,u32
"""ENSG00000103449""","[5.00362, 5.00362, … 5.00362]",100,100
"""ENSG00000132510""","[5.32903, 5.32903, … 5.32903]",100,100
"""ENSG00000182095""","[5.88375, 5.88375, … 5.88375]",100,100
"""ENSG00000197249""","[4.69934, 4.69934, … 4.69934]",100,100
"""ENSG00000168264""","[5.57258, 5.57258, … 0.0]",99,100
…,…,…,…
"""ENSG00000162592""","[4.23525, 0.0, … 0.0]",1,100
"""ENSG00000175040""","[0.0, 0.0, … 3.72381]",1,100
"""ENSG00000181781""","[0.0, 0.0, … 3.42903]",1,100


In [13]:
genes_w_H3K9ac_df.filter(pl.col('H3K9ac_len') < 100).sort(['H3K9ac_wc'], descending=True)

gene_id,H3K9ac,H3K9ac_wc,H3K9ac_len
str,list[f64],u32,u32


In [14]:
genes_w_H3K9me3_df.filter(pl.col('H3K9me3_wc') > 0).sort(['H3K9me3_wc'], descending=True)

gene_id,H3K9me3,H3K9me3_wc,H3K9me3_len
str,list[f64],u32,u32
"""ENSG00000177025""","[0.0, 0.0, … 4.62323]",96,100
"""ENSG00000186446""","[4.53664, 4.53664, … 4.11052]",90,100
"""ENSG00000167562""","[4.39961, 4.39961, … 0.0]",87,100
"""ENSG00000204920""","[4.12244, 4.12244, … 0.0]",86,100
"""ENSG00000197128""","[4.6052, 4.6052, … 0.0]",84,100
…,…,…,…
"""ENSG00000206557""","[0.0, 0.0, … 0.0]",1,100
"""ENSG00000213020""","[0.0, 0.0, … 0.0]",1,100
"""ENSG00000223638""","[0.0, 0.0, … 0.0]",1,100


In [15]:
genes_w_H3K9me3_df.filter(pl.col('H3K9me3_len') < 100).sort(['H3K9me3_wc'], descending=True)

gene_id,H3K9me3,H3K9me3_wc,H3K9me3_len
str,list[f64],u32,u32


In [16]:
genes_w_H3K4me3_df.filter(pl.col('H3K4me3_wc') > 0).sort(['H3K4me3_wc'], descending=True)

gene_id,H3K4me3,H3K4me3_wc,H3K4me3_len
str,list[f64],u32,u32
"""ENSG00000090612""","[4.71889, 4.71889, … 4.71889]",100,100
"""ENSG00000103343""","[4.31107, 4.31107, … 4.31107]",100,100
"""ENSG00000103449""","[5.62101, 5.62101, … 5.62101]",100,100
"""ENSG00000113248""","[3.97866, 3.97866, … 3.97866]",100,100
"""ENSG00000121413""","[4.12719, 4.12719, … 4.12719]",100,100
…,…,…,…
"""ENSG00000188676""","[0.0, 0.0, … 3.3125]",1,100
"""ENSG00000205041""","[0.0, 0.0, … 2.95537]",1,100
"""ENSG00000211584""","[0.0, 0.0, … 4.09985]",1,100


In [17]:
genes_w_H3K4me3_df.filter(pl.col('H3K4me3_wc') > 0).sort(['H3K4me3_wc'], descending=True)

gene_id,H3K4me3,H3K4me3_wc,H3K4me3_len
str,list[f64],u32,u32
"""ENSG00000090612""","[4.71889, 4.71889, … 4.71889]",100,100
"""ENSG00000103343""","[4.31107, 4.31107, … 4.31107]",100,100
"""ENSG00000103449""","[5.62101, 5.62101, … 5.62101]",100,100
"""ENSG00000113248""","[3.97866, 3.97866, … 3.97866]",100,100
"""ENSG00000121413""","[4.12719, 4.12719, … 4.12719]",100,100
…,…,…,…
"""ENSG00000188676""","[0.0, 0.0, … 3.3125]",1,100
"""ENSG00000205041""","[0.0, 0.0, … 2.95537]",1,100
"""ENSG00000211584""","[0.0, 0.0, … 4.09985]",1,100


In [18]:
genes_w_H3K4me3_df.filter(pl.col('H3K4me3_len') < 100).sort(['H3K4me3_wc'], descending=True)

gene_id,H3K4me3,H3K4me3_wc,H3K4me3_len
str,list[f64],u32,u32


In [19]:
genes_w_H3K27ac_df.filter(pl.col('H3K27ac_wc') > 0).sort(['H3K27ac_wc'], descending=True)

gene_id,H3K27ac,H3K27ac_wc,H3K27ac_len
str,list[f64],u32,u32
"""ENSG00000004399""","[4.73923, 4.73923, … 4.73923]",100,100
"""ENSG00000023839""","[5.43359, 5.43359, … 5.43359]",100,100
"""ENSG00000025708""","[15.6945, 15.6945, … 15.6945]",100,100
"""ENSG00000040633""","[10.535, 10.535, … 10.535]",100,100
"""ENSG00000047457""","[8.39445, 8.39445, … 8.39445]",100,100
…,…,…,…
"""ENSG00000185585""","[6.58023, 0.0, … 0.0]",1,100
"""ENSG00000188959""","[0.0, 0.0, … 3.13945]",1,100
"""ENSG00000213029""","[2.89758, 0.0, … 0.0]",1,100


In [20]:
genes_w_H3K27ac_df.filter(pl.col('H3K27ac_len') < 100).sort(['H3K27ac_wc'], descending=True)

gene_id,H3K27ac,H3K27ac_wc,H3K27ac_len
str,list[f64],u32,u32


In [21]:
genes_w_H3K27me3_df.filter(pl.col('H3K27me3_wc') > 0).sort(['H3K27me3_wc'], descending=True)

gene_id,H3K27me3,H3K27me3_wc,H3K27me3_len
str,list[f64],u32,u32
"""ENSG00000092607""","[4.29376, 4.29376, … 4.29376]",100,100
"""ENSG00000104375""","[3.96282, 3.96282, … 3.96282]",100,100
"""ENSG00000113196""","[4.31807, 4.31807, … 4.31807]",100,100
"""ENSG00000128714""","[4.33836, 4.33836, … 4.33836]",100,100
"""ENSG00000136352""","[4.04974, 4.04974, … 4.04974]",100,100
…,…,…,…
"""ENSG00000182646""","[0.0, 0.0, … 3.55038]",1,100
"""ENSG00000196408""","[0.0, 0.0, … 4.1525]",1,100
"""ENSG00000197263""","[0.0, 0.0, … 3.60476]",1,100


In [22]:
genes_w_H3K27me3_df.filter(pl.col('H3K27me3_len') < 100).sort(['H3K27me3_wc'], descending=True)

gene_id,H3K27me3,H3K27me3_wc,H3K27me3_len
str,list[f64],u32,u32


# Join all histones into single dataframe

In [23]:
# Join all histones into single dataframe
genes_histone_df = genes_w_H3K9ac_df \
                    .join(genes_w_H3K9me3_df, on='gene_id') \
                    .join(genes_w_H3K4me3_df, on='gene_id') \
                    .join(genes_w_H3K27ac_df, on='gene_id') \
                    .join(genes_w_H3K27me3_df, on='gene_id')

In [24]:
genes_histone_df

gene_id,H3K9ac,H3K9ac_wc,H3K9ac_len,H3K9me3,H3K9me3_wc,H3K9me3_len,H3K4me3,H3K4me3_wc,H3K4me3_len,H3K27ac,H3K27ac_wc,H3K27ac_len,H3K27me3,H3K27me3_wc,H3K27me3_len
str,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32
"""ENSG00000000003""","[0.0, 0.0, … 0.0]",9,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",17,100,"[0.0, 0.0, … 0.0]",23,100,"[0.0, 0.0, … 0.0]",0,100
"""ENSG00000000005""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100
"""ENSG00000000419""","[0.0, 0.0, … 0.0]",27,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",36,100,"[0.0, 0.0, … 0.0]",33,100,"[0.0, 0.0, … 0.0]",0,100
"""ENSG00000000457""","[0.0, 0.0, … 0.0]",27,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",37,100,"[0.0, 0.0, … 0.0]",42,100,"[0.0, 0.0, … 0.0]",0,100
"""ENSG00000000460""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",1,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENSG00000259658""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100
"""ENSG00000259664""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100
"""ENSG00000259680""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100


In [25]:
# Join genes with the value and histone features
genes_values_df = genes_df.select(['gene_id', 'value'])
genes_values_df

gene_id,value
str,f64
"""ENSG00000000003""",73.205
"""ENSG00000000005""",0.191
"""ENSG00000000419""",52.609
"""ENSG00000000457""",4.733
"""ENSG00000000460""",0.942
…,…
"""ENSG00000259658""",0.212
"""ENSG00000259664""",0.0
"""ENSG00000259680""",0.071


In [26]:
genes_histone_values_df = genes_histone_df.join(
    genes_values_df,
    on = 'gene_id'
)

In [27]:
genes_histone_values_df

gene_id,H3K9ac,H3K9ac_wc,H3K9ac_len,H3K9me3,H3K9me3_wc,H3K9me3_len,H3K4me3,H3K4me3_wc,H3K4me3_len,H3K27ac,H3K27ac_wc,H3K27ac_len,H3K27me3,H3K27me3_wc,H3K27me3_len,value
str,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,f64
"""ENSG00000000003""","[0.0, 0.0, … 0.0]",9,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",17,100,"[0.0, 0.0, … 0.0]",23,100,"[0.0, 0.0, … 0.0]",0,100,73.205
"""ENSG00000000005""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.191
"""ENSG00000000419""","[0.0, 0.0, … 0.0]",27,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",36,100,"[0.0, 0.0, … 0.0]",33,100,"[0.0, 0.0, … 0.0]",0,100,52.609
"""ENSG00000000457""","[0.0, 0.0, … 0.0]",27,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",37,100,"[0.0, 0.0, … 0.0]",42,100,"[0.0, 0.0, … 0.0]",0,100,4.733
"""ENSG00000000460""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",1,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.942
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENSG00000259658""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.212
"""ENSG00000259664""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.0
"""ENSG00000259680""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.071


In [28]:
# Save to parquet
genes_histone_values_df.write_parquet(os.path.join(DATASET_PATH, 'donor3_exp_histones.parquet'))

In [30]:
test_df = pl.read_parquet(os.path.join(DATASET_PATH, 'donor3_exp_histones.parquet'))
test_df

gene_id,H3K9ac,H3K9ac_wc,H3K9ac_len,H3K9me3,H3K9me3_wc,H3K9me3_len,H3K4me3,H3K4me3_wc,H3K4me3_len,H3K27ac,H3K27ac_wc,H3K27ac_len,H3K27me3,H3K27me3_wc,H3K27me3_len,value
str,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,f64
"""ENSG00000000003""","[0.0, 0.0, … 0.0]",9,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",17,100,"[0.0, 0.0, … 0.0]",23,100,"[0.0, 0.0, … 0.0]",0,100,73.205
"""ENSG00000000005""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.191
"""ENSG00000000419""","[0.0, 0.0, … 0.0]",27,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",36,100,"[0.0, 0.0, … 0.0]",33,100,"[0.0, 0.0, … 0.0]",0,100,52.609
"""ENSG00000000457""","[0.0, 0.0, … 0.0]",27,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",37,100,"[0.0, 0.0, … 0.0]",42,100,"[0.0, 0.0, … 0.0]",0,100,4.733
"""ENSG00000000460""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",1,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.942
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENSG00000259658""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.212
"""ENSG00000259664""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.0
"""ENSG00000259680""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.071
